# Dataset Blend & Chinchilla Analysis

- Token count estimates from mmap files
- 70/30 general/code blend visualization
- Chinchilla compute-optimal analysis for each model scale
- Data budget planning for scaling runs

In [ ]:
import struct
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

DATA_DIR      = Path("/data")
TOKENIZED_DIR = DATA_DIR / "curated/tokenized"

## Token Count from mmap Files

In [ ]:
def read_idx(idx_path: Path) -> tuple[int, int, float, float]:
    """Read doc count, token count, min/max doc length from .idx file."""
    with open(idx_path, "rb") as f:
        f.read(9 + 8 + 1)  # header
        doc_count = struct.unpack("<Q", f.read(8))[0]
        sizes = np.frombuffer(f.read(doc_count * 4), dtype=np.int32)
    return doc_count, int(sizes.sum()), float(sizes.min()), float(sizes.max())


idx_files = sorted(TOKENIZED_DIR.glob("*.idx"))

if idx_files:
    total_docs = 0
    total_tokens = 0
    print(f"{'File':<40} {'Docs':>10} {'Tokens':>14} {'Min len':>10} {'Max len':>10}")
    print("-" * 90)
    for idx in idx_files:
        docs, tokens, min_len, max_len = read_idx(idx)
        total_docs   += docs
        total_tokens += tokens
        print(f"  {idx.name:<40} {docs:>10,} {tokens:>14,} {min_len:>10.0f} {max_len:>10.0f}")
    print("-" * 90)
    print(f"  {'TOTAL':<40} {total_docs:>10,} {total_tokens:>14,}")
    print(f"\n  Total: {total_tokens/1e9:.3f}B tokens")
else:
    print("No tokenized files found. Run 'make tokenizer' first.")
    total_tokens = 0

## Chinchilla Compute-Optimal Analysis

Chinchilla (Hoffmann et al., 2022) shows the optimal token count
is approximately **20× the model parameter count**.

In [ ]:
# Model scales and their Chinchilla-optimal token targets
scales = [
    {"name": "125M",  "params": 125e6,  "target_tokens": 2.5e9,  "gpus": 1,  "color": "#378ADD"},
    {"name": "350M",  "params": 350e6,  "target_tokens": 7.0e9,  "gpus": 4,  "color": "#7F77DD"},
    {"name": "1B",    "params": 1.0e9,  "target_tokens": 20.0e9, "gpus": 4,  "color": "#D85A30"},
]

print(f"{'Scale':<8} {'Params':>10} {'Chinchilla target':>20} {'Available':>15} {'Coverage':>12}")
print("-" * 70)
for s in scales:
    chinchilla = s["params"] * 20
    coverage = total_tokens / chinchilla * 100 if total_tokens > 0 else 0
    print(
        f"  {s['name']:<8} {s['params']/1e6:>8.0f}M "
        f"{chinchilla/1e9:>18.1f}B "
        f"{total_tokens/1e9:>13.2f}B "
        f"{coverage:>11.1f}%"
    )

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 5))

param_range = np.logspace(7, 10, 200)
chinchilla_tokens = param_range * 20
ax.plot(param_range / 1e6, chinchilla_tokens / 1e9,
        color="#888780", linewidth=1.5, linestyle="--", label="Chinchilla optimal (20× params)")

for s in scales:
    ax.scatter(s["params"] / 1e6, s["target_tokens"] / 1e9,
               color=s["color"], s=120, zorder=5, label=s["name"])
    ax.annotate(
        f"{s['name']}\n{s['target_tokens']/1e9:.1f}B tokens",
        (s["params"] / 1e6, s["target_tokens"] / 1e9),
        textcoords="offset points", xytext=(10, 5), fontsize=9, color=s["color"]
    )

if total_tokens > 0:
    ax.axhline(total_tokens / 1e9, color="#1D9E75", linewidth=1.5,
               linestyle="-.", label=f"Available ({total_tokens/1e9:.2f}B tokens)")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Model parameters (M)")
ax.set_ylabel("Training tokens (B)")
ax.set_title("Chinchilla compute-optimal scaling — SLM pipeline")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, which="both")
plt.tight_layout()
plt.savefig("/results/eval/chinchilla_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## Data Blend Visualization

In [ ]:
# The pre-training config blends: 70% general text, 30% code
blend = {
    "General text\n(Common Crawl + Wikipedia)": 0.70,
    "Code\n(CodeSearchNet Python)": 0.30,
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie(
    list(blend.values()),
    labels=list(blend.keys()),
    colors=["#5DCAA5", "#AFA9EC"],
    autopct="%1.0f%%",
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
axes[0].set_title("Pre-training data blend")

# SFT data sources
sft_sources = {
    "OASST1": 8000,
    "Dolly 15k": 15000,
    "CodeSearchNet\n(write)": 100000,
    "CodeSearchNet\n(explain)": 100000,
}
colors_sft = ["#378ADD", "#85B7EB", "#AFA9EC", "#CECBF6"]
axes[1].bar(list(sft_sources.keys()), list(sft_sources.values()),
            color=colors_sft, edgecolor="white")
axes[1].set_ylabel("Examples")
axes[1].set_title("SFT dataset sizes (approximate)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

plt.tight_layout()
plt.savefig("/results/eval/dataset_blend.png", dpi=150, bbox_inches="tight")
plt.show()

## WARCs Needed for Scaling

In [ ]:
# Estimate WARC files needed per scale
# From our run: 20 WARCs → ~14.8k docs → estimate tokens after full pipeline
# Adjust these based on your actual run results
TOKENS_PER_WARC_ESTIMATE = total_tokens / 20 if total_tokens > 0 else 50e6

print(f"Estimated tokens per WARC: {TOKENS_PER_WARC_ESTIMATE/1e6:.0f}M")
print()
print(f"{'Scale':<8} {'Target tokens':>16} {'WARCs needed':>14} {'Download size':>15}")
print("-" * 58)
for s in scales:
    warcs = int(np.ceil(s["target_tokens"] / TOKENS_PER_WARC_ESTIMATE))
    dl_gb = warcs * 1.1  # ~1.1GB per WARC compressed
    print(f"  {s['name']:<8} {s['target_tokens']/1e9:>14.1f}B {warcs:>14,} {dl_gb:>13.0f}GB")